In [1]:
import csv
import json
import shutil
from pathlib import Path
from collections import Counter

# Locate the dataset
DATA_FILE = Path("user_behavior_dataset.csv")

if not DATA_FILE.exists():
    raise FileNotFoundError("user_behavior_dataset.csv was not found.")

# Read and inspect the dataset
with open(DATA_FILE, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    print("COLUMN NAMES:")
    print(reader.fieldnames)

    rows = list(reader)

print("\nDATASET INFORMATION")
print("Total records:", len(rows))

print("\nFIRST 5 RECORDS:")
for row in rows[:5]:
    print(row)

COLUMN NAMES:
['User ID', 'Device Model', 'Operating System', 'App Usage Time (min/day)', 'Screen On Time (hours/day)', 'Battery Drain (mAh/day)', 'Number of Apps Installed', 'Data Usage (MB/day)', 'Age', 'Gender', 'User Behavior Class']

DATASET INFORMATION
Total records: 700

FIRST 5 RECORDS:
{'User ID': '1', 'Device Model': 'Google Pixel 5', 'Operating System': 'Android', 'App Usage Time (min/day)': '393', 'Screen On Time (hours/day)': '6.4', 'Battery Drain (mAh/day)': '1872', 'Number of Apps Installed': '67', 'Data Usage (MB/day)': '1122', 'Age': '40', 'Gender': 'Male', 'User Behavior Class': '4'}
{'User ID': '2', 'Device Model': 'OnePlus 9', 'Operating System': 'Android', 'App Usage Time (min/day)': '268', 'Screen On Time (hours/day)': '4.7', 'Battery Drain (mAh/day)': '1331', 'Number of Apps Installed': '42', 'Data Usage (MB/day)': '944', 'Age': '47', 'Gender': 'Female', 'User Behavior Class': '3'}
{'User ID': '3', 'Device Model': 'Xiaomi Mi 11', 'Operating System': 'Android', 'A

In [2]:

# STUDENT-SPECIFIC FILTERING CRITERIA
ID_SUM = 25
SEED_DIGIT = 4

# Determine the assigned operating system and behavior class
TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1

# Filter records by operating system
os_records = [
    row for row in rows
    if row["Operating System"] == TARGET_OS
]

# Filter records by target behavior class
selected_records = [
    row for row in os_records
    if int(row["User Behavior Class"]) == TARGET_BEHAVIOR_CLASS
]

# Display filtering results
print("FILTERING RESULTS")
print("-----------------")
print("Student ID Sum:", ID_SUM)
print("Seed Digit:", SEED_DIGIT)
print("Assigned Operating System:", TARGET_OS)
print("Target Behavior Class:", TARGET_BEHAVIOR_CLASS)
print("Total records in dataset:", len(rows))
print("Records matching operating system:", len(os_records))
print("Records in target behavior class:", len(selected_records))


FILTERING RESULTS
-----------------
Student ID Sum: 25
Seed Digit: 4
Assigned Operating System: Android
Target Behavior Class: 5
Total records in dataset: 700
Records matching operating system: 554
Records in target behavior class: 105


In [3]:

# CALCULATE AVERAGE BATTERY DRAIN AND DATA USAGE

if selected_records:
    total_battery_drain = sum(
        int(row["Battery Drain (mAh/day)"])
        for row in selected_records
    )

    total_data_usage = sum(
        int(row["Data Usage (MB/day)"])
        for row in selected_records
    )

    average_battery_drain = total_battery_drain / len(selected_records)
    average_data_usage = total_data_usage / len(selected_records)

else:
    average_battery_drain = 0
    average_data_usage = 0

# Display calculation results
print("USAGE PROFILE SUMMARY")
print("---------------------")
print("Selected users:", len(selected_records))
print("Average battery drain:", round(average_battery_drain, 2), "mAh/day")
print("Average data usage:", round(average_data_usage, 2), "MB/day")


USAGE PROFILE SUMMARY
---------------------
Selected users: 105
Average battery drain: 2693.06 mAh/day
Average data usage: 1984.77 MB/day


In [5]:

# EXPORT SELECTED RECORDS TO CSV

EXPORT_FIELDS = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)"
]

CSV_FILE = Path("filtered_users.csv")

with open(CSV_FILE, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=EXPORT_FIELDS,
        extrasaction="ignore"
    )

    writer.writeheader()
    writer.writerows(selected_records)

print("CSV EXPORT RESULTS")
print("------------------")
print("Exported file:", CSV_FILE.name)
print("Exported records:", len(selected_records))
print("Exported fields:", EXPORT_FIELDS)
print("File successfully created:", CSV_FILE.exists())


CSV EXPORT RESULTS
------------------
Exported file: filtered_users.csv
Exported records: 105
Exported fields: ['User ID', 'Device Model', 'Operating System', 'Battery Drain (mAh/day)', 'Data Usage (MB/day)']
File successfully created: True


In [6]:

# CREATE JSON SUMMARY AND VERIFY CSV EXPORT

JSON_FILE = Path("usage_summary.json")

# Count exported CSV records
with open(CSV_FILE, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    exported_rows = list(reader)

exported_count = len(exported_rows)

# Verify exported record count
count_verified = exported_count == len(selected_records)

# Create JSON summary
summary = {
    "Student ID Sum": ID_SUM,
    "Seed Digit": SEED_DIGIT,
    "Operating System": TARGET_OS,
    "User Behavior Class": TARGET_BEHAVIOR_CLASS,
    "Total Dataset Records": len(rows),
    "Records Matching Operating System": len(os_records),
    "Records in Target Behavior Class": len(selected_records),
    "Average Battery Drain (mAh/day)": round(average_battery_drain, 2),
    "Average Data Usage (MB/day)": round(average_data_usage, 2),
    "Exported CSV Records": exported_count,
    "Count Verification": count_verified
}

# Write JSON summary file
with open(JSON_FILE, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

# Display verification results
print("JSON SUMMARY AND VERIFICATION")
print("-----------------------------")
print("JSON file:", JSON_FILE.name)
print("Selected records:", len(selected_records))
print("Exported CSV records:", exported_count)
print("CSV count verified:", count_verified)
print("JSON file successfully created:", JSON_FILE.exists())


JSON SUMMARY AND VERIFICATION
-----------------------------
JSON file: usage_summary.json
Selected records: 105
Exported CSV records: 105
CSV count verified: True
JSON file successfully created: True


In [7]:

# CREATE CSV BACKUP AND CHECK FILE SIZE

BACKUP_FILE = Path("filtered_users_backup.csv")

# Create a backup copy of the exported CSV
shutil.copy2(CSV_FILE, BACKUP_FILE)

# Get file sizes in bytes
csv_size = CSV_FILE.stat().st_size
backup_size = BACKUP_FILE.stat().st_size

# Verify backup file
backup_verified = (
    BACKUP_FILE.exists()
    and CSV_FILE.read_bytes() == BACKUP_FILE.read_bytes()
)

# Display backup results
print("BACKUP AND FILE SIZE RESULTS")
print("----------------------------")
print("Original CSV:", CSV_FILE.name)
print("Backup CSV:", BACKUP_FILE.name)
print("Original CSV size:", csv_size, "bytes")
print("Backup CSV size:", backup_size, "bytes")
print("Backup successfully created:", BACKUP_FILE.exists())
print("Backup verified:", backup_verified)


BACKUP AND FILE SIZE RESULTS
----------------------------
Original CSV: filtered_users.csv
Backup CSV: filtered_users_backup.csv
Original CSV size: 3959 bytes
Backup CSV size: 3959 bytes
Backup successfully created: True
Backup verified: True
